<a href="https://colab.research.google.com/github/SURYA-CREATE-ops/dotnet-project-1/blob/master/Copy_of_integrating_mcp_server_in_skillmap_agent_initial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AI Model's Instance Creation

In [ ]:
!pip install -U langchain-google-genai

In [ ]:
!pip install langchain langchain-tavily

In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

google_api_key = userdata.get('GEMINI_API_KEY')
model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)

In [ ]:
!pip install langchain-mcp-adapters

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient


In [ ]:
COMPOSE_API_KEY = userdata.get('COMPOSE_API_KEY')

In [ ]:
client = MultiServerMCPClient(
    {
        "mcp_tavily": {
            "transport": "http",
            "url": "https://backend.composio.dev/v3/mcp/b827882d-eb98-491d-82d1-53f70423271c/mcp?user_id=pg-test-7656991e-a28f-4be4-a413-e7e3a74d48a6",
            "headers" : {"x-api-key": COMPOSE_API_KEY}
        }
    }
)

pg-test-7656991e-a28f-4be4-a413-e7e3a74d48a6

TAVILY TOOL CREATION

In [ ]:
from langchain_tavily import TavilySearch
from google.colab import userdata
tavily_api_key = userdata.get('TAVILY_API_KEY')
skill_demand_tool = TavilySearch(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=tavily_api_key,
)

Rapid Api website's JSearch for getting Live Job Links

In [ ]:
import requests
from langchain.tools import tool
from google.colab import userdata

@tool
def search_jobs(skill: str, location: str) -> list:
  """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
  print(f"\nCalling search_jobs tool")
  print(f"Searching jobs for: {skill} in {location}")

  rapidapi_key = userdata.get('RAPID_API')

  url = "https://jsearch.p.rapidapi.com/search"
  headers = {
    "x-rapidapi-key": rapidapi_key,
    "x-rapidapi-host": "jsearch.p.rapidapi.com"
  }
  querystring = {
    "query": f"{skill} in {location}",
    "page": "1",
    "country": "in",
    "employment_types": "INTERN,FULLTIME",
    "job_requirements": "no_experience,under_3_years_experience"
  }
  response = requests.get(url, headers=headers, params=querystring)
  data = response.json()
  jobs = data.get("data", [])
  print(f"Found {len(jobs)} jobs\n")

  result = []
  for job in jobs:
    result.append({
      "title": job.get("job_title"),
      "company": job.get("employer_name"),
      "location": job.get("job_city"),
      "apply_link": job.get("job_apply_link")
    })
  return result


In [ ]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- search tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""


In [ ]:
from langchain.agents import create_agent

async def skill_map_agent():
    # Create agent
    # mcp_tools = await client.get_tools()
    # all_tools = mcp_tools + [search_jobs]
    all_tools = [skill_demand_tool, search_jobs]

    agent = create_agent(
        model=model,
        tools=all_tools,
        debug=True,
        system_prompt= system_prompt

    )


    # User query
    user_query = "What's the demand for generative AI in the industry and show me related job openings in India"

    # Invoke agent
    response = await agent.ainvoke({
        "messages": [
            {"role": "user", "content": user_query}
        ]
    })

    # Print response
    print(response["messages"][-1].content[0]["text"])


# Run function
await skill_map_agent()

[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry and show me related job openings in India", additional_kwargs={}, response_metadata={}, id='922be7c2-88f2-4a9b-a76e-a1cdef831eb6')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_jobs', 'arguments': '{"skill": "generative AI", "location": "India"}'}, '__gemini_function_call_thought_signatures__': {'1fce5839-385a-4e6d-af22-c01c343d7d4f': 'CqIEAQw51scPLAqxuPy7oDnZJGuTk4Uaa4HqiRl7yHxUalyYQla53uajhl4AAx/QA+dbbZLTb2oIa+ITGTtDaf8/czM9sFZZlZ4s5ur0xmEBq3lksBy573PXUzNmJ0ZfpIB63yPJZXdBgvpAfvYr97zl3WaXwFGm/mQ0qzS4pUU1weexxEybhCznK8XlkvN4hmjjns169NN4qvpzJWbYW/3emoN188086/ZcVgTwhp+p2aeWif7d4Ajbe5eH4KdeY3dQ3/FGDMvTHptDIkG8Ug2Mau0Rsl6ZxZjepvrRsMcBpMzwjKB+4atHCFy+WQDQiAQFgpY4r/Xgn+bvt5eYfCMuED7tD0NoZnU44ngCG35VXJ9SOpoOCjxN0d17HZ6/v8XvNdpxxPgF09E3VKA78LsZ/EDL0O3A2fm7Fz8NnuPBe+dgdAGqmBk6CAxS1qmkCq5Gv+O/fqqVGVmJBUXpZbaief8ScjAPKwiC4NZpFD0tK4E9+Cdp

from langchain.agents import create_agent

def skill_map_agent():

  agent = create_agent(
    model=model,
  tools=[skill_demand_tool, search_jobs],
  system_prompt=system_prompt,
  debug=True

  )
  user_query = "What's the demand for generative ai in the industry and show me related job openings in India"

  response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
  })
  print(response["messages"][-1].content[0]["text"])


skill_map_agent()